# Head-Only Detector — Hybrid Architecture Companion

Trains a YOLOv8n single-class **head** detector on the *OverHead Head
Detection* dataset (top-down camera angle — matches the cabin CCTV
scenario). The resulting `best_head.pt` is plugged into
`scripts/run_simulation.py` via `--head-weights` for the hybrid
two-model occupancy estimator described in the thesis methodology.

## Why a separate head model?

Single-frame elevator CCTV exhibits two systematic failure modes for
the four-class detector:

1. **Body occlusion in crowded cabins** — closely packed passengers
   share a visual silhouette so the detector merges or drops boxes.
2. **Top-down class confusion** — overhead poses look geometrically
   similar to luggage / boxes.

A head-only detector is robust to both: heads are always at the top
of every cabin occupant and remain visible regardless of crowding.

## Inputs expected on Drive

Place these in `MyDrive/Capstone/`:

| File | Source |
|---|---|
| `overhead_head.zip` | Rename `OverHead Head Detection.yolov8.zip` to `overhead_head.zip` and upload |
| `05_head_model_training.ipynb` | This notebook |

## Outputs written to Drive

* `MyDrive/Capstone/models/runs/elevator_head_v1/` — training run
  artifacts (loss curves, PR / F1 plots, confusion matrix)
* `MyDrive/Capstone/models/weights/best_head.pt` — production checkpoint

## 1. Mount Drive and prepare workspace

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile
from pathlib import Path

BASE_DRIVE = '/content/drive/MyDrive/Capstone'
WORK = '/content/work'
DATA_DIR = f'{WORK}/data/overhead_head'

ZIP_PATH = f'{BASE_DRIVE}/overhead_head.zip'
assert os.path.exists(ZIP_PATH), (
    f'overhead_head.zip not found at {ZIP_PATH}. Upload it to MyDrive/Capstone/.'
)

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)
print('Extracted to:', DATA_DIR)
%cd {DATA_DIR}
!ls

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics

## 3. Consolidate labels — collapse all classes to a single `head` class

The downloaded dataset has four classes
(`head`, `head-top-view`, `people`, `person`). Our hybrid pipeline
only needs *one* head class, so we rewrite every label file to use
class id `0` and overwrite `data.yaml` with a single-class spec.

This keeps the bounding boxes intact (they all describe heads or
head-shoulder regions) while turning the problem into a single-class
detection task — easier to train and easier to ensemble at inference.

In [ ]:
from pathlib import Path
import yaml

data_yaml_path = Path(DATA_DIR) / 'data.yaml'
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)
print('Original classes:', cfg.get('names'))

# Rewrite every label so class id is always 0.
rewritten = 0
for split in ('train', 'valid', 'val', 'test'):
    lbl_dir = Path(DATA_DIR) / split / 'labels'
    if not lbl_dir.is_dir():
        continue
    for lbl in lbl_dir.glob('*.txt'):
        new_lines = []
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                new_lines.append(' '.join(['0'] + parts[1:]))
        with open(lbl, 'w') as f:
            f.write('\n'.join(new_lines))
        rewritten += 1
print(f'Rewrote {rewritten} label files to single class.')

# Overwrite data.yaml with a single-class spec.
cfg['names'] = ['head']
cfg['nc'] = 1
cfg['path'] = str(data_yaml_path.parent)
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('\nNew data.yaml:')
print(open(data_yaml_path).read())

## 4. GPU + dataset audit

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi 2>/dev/null | head -10

for split in ('train', 'valid', 'test'):
    img_dir = Path(DATA_DIR) / split / 'images'
    if img_dir.is_dir():
        n = sum(1 for _ in img_dir.iterdir())
        print(f'  {split:<6}: {n} images')

## 5. Train

Hyperparameters tuned for an L4 GPU and a small ~6k single-class
dataset. Expected wall-time: **~1.5 – 2.5 hours** for 80 epochs (early
stopping typically fires around epoch 40–60).

In [ ]:
from ultralytics import YOLO

VARIANT = 'yolov8n.pt'
EPOCHS  = 80
BATCH   = 32
IMGSZ   = 640

RUNS_DIR = f'{BASE_DRIVE}/models/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(VARIANT)
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=0.01,
    patience=20,
    seed=42,
    project=RUNS_DIR,
    name='elevator_head_v1',
    plots=True,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
)
BEST_HEAD = f'{results.save_dir}/weights/best.pt'
print('best head checkpoint:', BEST_HEAD)

## 6. Evaluate on the held-out test split

In [ ]:
model_head = YOLO(BEST_HEAD)
metrics = model_head.val(data=str(data_yaml_path), split='test')
print('mAP50:    ', metrics.box.map50)
print('mAP50-95: ', metrics.box.map)

## 7. Save the head checkpoint to Drive

`best_head.pt` lives next to `best.pt` (the four-class checkpoint).
The hybrid simulation script picks them up via `--weights` and
`--head-weights`.

In [ ]:
import shutil
drive_dst = f'{BASE_DRIVE}/models/weights/best_head.pt'
os.makedirs(os.path.dirname(drive_dst), exist_ok=True)
shutil.copy2(BEST_HEAD, drive_dst)
print('copied:', drive_dst, f'({os.path.getsize(drive_dst) / 1e6:.1f} MB)')

## 8. Smoke test on a couple of cabin frames

Optional. Drop one or two cabin photos into `/content/work/probe/` and
re-run the cell to visualize how many heads the new model finds.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

probe_dir = f'{WORK}/probe'
os.makedirs(probe_dir, exist_ok=True)
probe_imgs = sorted(
    glob.glob(f'{probe_dir}/*.jpg') + glob.glob(f'{probe_dir}/*.png')
)[:6]
if not probe_imgs:
    print(f'Drop a few sample images at {probe_dir}/ then re-run this cell.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, p in zip(axes.flat, probe_imgs):
        res = model_head.predict(p, conf=0.25, verbose=False)[0]
        n_heads = 0 if res.boxes is None else len(res.boxes)
        ax.imshow(Image.fromarray(res.plot()[..., ::-1]))
        ax.set_title(f'{os.path.basename(p)} — {n_heads} heads', fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 9. What to do back on the local machine

1. Drive will sync `best_head.pt` to your local Capstone folder.
   Otherwise download it manually from
   `MyDrive/Capstone/models/weights/best_head.pt` and place it under
   `models/weights/best_head.pt` in the project.
2. Re-run the energy simulation in **hybrid mode**:

   ```
   python -m scripts.run_simulation \
       --images data/sim/images \
       --ground-truth data/sim/ground_truth.csv \
       --weights models/weights/best.pt \
       --head-weights models/weights/best_head.pt \
       --rated-capacity 8 \
       --num-calls 1000 \
       --output results/simulation/hybrid
   ```

3. Compare `results/simulation/baseline/` and
   `results/simulation/hybrid/` to populate the ablation table for the
   thesis Results chapter.